# aw_02_a1 — Stage A1: PlayWorld SFT (Track A direct)

**Protocol**: §5.1 A1. Input: G3 frozen suites. Output: A1 adapter run + eval pair + analysis.

**Result summary (v0.4.x)**: A1 beats base on all 5 suites
(pass-rate Δ: adversarial +0.443, template/rule-OOD +0.193, ID +0.187, comp-OOD +0.120;
all paired-bootstrap CIs exclude 0, permutation p ≤ 0.0003).

**Failure-analysis narrative (archived below as x01–x08)**: the first A1 eval collapsed
(93% malformed_json) despite 95.5% train accuracy. An 8-step diagnostic chain
(template → adapter loading → truncation → completion learning → boundary tokens →
TRL rendering → trainer labels → dynamics/checkpoint) localized the root cause:
Qwen3's chat template injects an empty `<think>\n\n</think>\n\n` opener into every
assistant target, but Qwen3-8B-BASE has untrained think-token embeddings that
attention/MLP LoRA cannot repair. Fix: seed the opener at eval time
(`evaluation.opener_seed`, amendment v1.1) — no retrain needed.


In [ ]:
# @title common header
import os
import sys
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
os.environ["WANDB_API_KEY"] = userdata.get('WANDB_API_KEY')
os.environ["GITHUB_TOKEN"] = userdata.get('GITHUB_TOKEN')

!git clone https://{os.environ["GITHUB_TOKEN"]}@github.com/m97j/axiom-world.git
%cd axiom-world
!pip install -e . -r requirements/colab-g4.lock.txt


Cloning into 'axiom-world'...
remote: Enumerating objects: 387, done.
remote: Counting objects: 100% (387/387), done.
remote: Compressing objects: 100% (250/250), done.
remote: Total 387 (delta 187), reused 310 (delta 110), pack-reused 0 (from 0)
Receiving objects: 100% (387/387), 141.04 KiB | 7.83 MiB/s, done.
Resolving deltas: 100% (187/187), done.
/content/axiom-world
Obtaining file:///content/axiom-world
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 885.0/885.0 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 74.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 141.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 68.0 MB/s eta 0:00:00
  Building editable for axiom-world (pyproject.toml) ... done

In [ ]:
# @title a_a1_data — build SFT + prompt data (leakage-gated)
!python scripts/build_eval_suites.py --episodes-per-suite 300 \
  --hf-sync-repo m97j/aw-playworld
!python scripts/build_training_data.py \
  --hf-sync-repo m97j/aw-posttrain

eval_id: 300 episodes -> data/eval_suites/eval_id.jsonl (sha256:aceeea727d2b9eaed...)
eval_template_ood: 300 episodes -> data/eval_suites/eval_template_ood.jsonl (sha256:13580a6cbf7a4e566...)
eval_comp_ood: 300 episodes -> data/eval_suites/eval_comp_ood.jsonl (sha256:444191a244dcd77d1...)
eval_rule_ood: 300 episodes -> data/eval_suites/eval_rule_ood.jsonl (sha256:d73745108f5e7e207...)
eval_adversarial: 300 episodes -> data/eval_suites/eval_adversarial.jsonl (sha256:c73dd155acd069292...)

G3 freeze manifest -> data/eval_suites/freeze_manifest.json
Commit this manifest; training loaders must pass eval_family_ids as forbidden_family_ids (leakage gate).
Found 6 files to upload
  Preparing   ████████████████████  6 / 6 ✓
  Uploading   ████████████████████  -
  Committing  ████████████████████  6 / 6 ✓
No files have been modified since last commit. Skipping to prevent empty commit.
eval suites persisted: hf://dataset/m97j/aw-playworld/eval_suites/v1
{
  "seed": 1042,
  "sft_records": 2000,
 

In [ ]:
# @title b_a1_train — A1 PlayWorld SFT
!python scripts/run_experiment.py \
  --config configs/experiments/a1_playworld_sft.yaml \
  --override data.source.local_path=data/train/playworld_sft.jsonl \
  --hf-sync-repo m97j/aw-runs-a1

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
run_id: 20260801-030335--a1-playworld-sft--s42--e24d72
model.safetensors.index.json: 100% 32.9k/32.9k [00:00<00:00, 80.7MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0% 0/5 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/8.39G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.00/16.4G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/16.4G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/16.4G [00:00<?, ?B/s]
Reconstructing (incomplete total...):  14% 2.32G/16

In [ ]:
# @title c_a1_eval — A1 adapter, batched greedy (canonical profile)
!python scripts/build_eval_suites.py --episodes-per-suite 300

RUN_ID = "20260801-030335--a1-playworld-sft--s42--e24d72"
out = !python scripts/fetch_run.py --repo m97j/aw-runs-a1 --run-id {RUN_ID}
print("\n".join(out))
adapter_dir = [l for l in out if l.startswith("ADAPTER_DIR=")][0].split("=", 1)[1]

!python scripts/run_evaluation.py \
  --config configs/experiments/eval_playworld.yaml \
  --adapter-dir {adapter_dir} \
  --max-new-tokens 1024 --batch-size 100 \
  --hf-sync-repo m97j/aw-runs-a1



eval_id: 300 episodes -> data/eval_suites/eval_id.jsonl (sha256:aceeea727d2b9eaed...)
eval_template_ood: 300 episodes -> data/eval_suites/eval_template_ood.jsonl (sha256:13580a6cbf7a4e566...)
eval_comp_ood: 300 episodes -> data/eval_suites/eval_comp_ood.jsonl (sha256:444191a244dcd77d1...)
eval_rule_ood: 300 episodes -> data/eval_suites/eval_rule_ood.jsonl (sha256:d73745108f5e7e207...)
eval_adversarial: 300 episodes -> data/eval_suites/eval_adversarial.jsonl (sha256:c73dd155acd069292...)

G3 freeze manifest -> data/eval_suites/freeze_manifest.json
Commit this manifest; training loaders must pass eval_family_ids as forbidden_family_ids (leakage gate).


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            


Fetching 14 files:   0% 0/14 [00:00<?, ?it/s]

Reconstructing (incomplete total...):   0% 0.00/11.4M [00:00<?, ?B/s]         

Reconstructing (incomplete total...):   0% 0.00/186M [00:00<?, ?B/s] 

Reconstructing (incomplete total...):   0% 0.00/186M [00:00<?

In [ ]:
# @title d_a1_eval_control — raw base model
!python scripts/run_evaluation.py \
  --config configs/experiments/eval_playworld.yaml \
  --max-new-tokens 1024 --batch-size 100 \
  --hf-sync-repo m97j/aw-runs-a1

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Loading weights: 100% 399/399 [00:01<00:00, 334.88it/s]
generate(batched): 100% 3/3 [06:12<00:00, 124.32s/it]
eval_adversarial: pass_rate={'mean': 0.15, 'ci95': [0.11, 0.1933]}
generate(batched): 100% 3/3 [04:38<00:00, 92.99s/it]
eval_comp_ood: pass_rate={'mean': 0.0033, 'ci95': [0.0, 0.01]}
generate(batched): 100% 3/3 [02:46<00:00, 55.44s/it]
eval_id: pass_rate={'mean': 0.0, 'ci95': [0.0, 0.0]}
generate(batched): 100% 3/3 [00:48<00:00, 16.31s/it]
eval_rule_ood: pass_rate={'mean': 0.0, 'ci95': [0.0, 0.0]}
generate(batched): 100% 3/3 [00:55<00:00, 18.46s/it]
eval_template_ood: pass_ra

In [ ]:
# @title f_a1_analysis — paired comparison (bootstrap CI + permutation)
RUN_ID_a = "20260801-063425--eval-playworld--s42--3bf440"
RUN_ID_b = "20260801-071014--eval-playworld--s42--b7aed3"

!python scripts/fetch_run.py --repo m97j/aw-runs-a1 --run-id {RUN_ID_a} --kind eval
!python scripts/fetch_run.py --repo m97j/aw-runs-a1 --run-id {RUN_ID_b} --kind eval

!python scripts/run_analysis.py \
    --run-a runs/{RUN_ID_a} --label-a a1-sft \
    --run-b runs/{RUN_ID_b} --label-b qwen3-8b-base \
    --output runs/{RUN_ID_a}/analysis_summary.json \
    --hf-sync-repo m97j/aw-runs-a1


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0% 0/8 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/1.90M [00:00<?, ?B/s]         
Reconstructing (incomplete total...): 100% 1.90M/1.90M [00:00<00:00, 4.69MB/s]
Reconstructing (incomplete total...):  47% 1.90M/4.01M [00:00<00:00, 4.69MB/s]
Reconstructing (incomplete total...):  47% 1.90M/4.01M [00:00<00:00, 4.69MB/s]

Fetching 8 files:  12% 1/8 [00:00<00:03,  2.18it/s]
Reconstructing (incomplete total...):  32% 1.90M/5.89M [00:00<00:00, 4.69MB/s]
Reconstructing (incomplete total...):  25% 1.90M/7.53M [00:00<00:01, 4.69MB/s]
Reconstructing (incomplete total...): 100% 7.53M/7.53M [00:00<00:00, 7.47MB/s]
Reconstructing (incomplete total...):  81% 7.53M/9.35M [00:00<00:00, 7.47MB/s]

Fetching 8 files: 100% 8/8 [00:00<00:00, 12.23it/s]
Download complete: 100% 9.35M/9.35M [00:00<00:00, 18.9MB/s]
Reconstruction complete: 100% 9.35M/9.35M [00:00<00:00, 18.9MB/s]             

---
## Failure Analysis (archived) — diagnostic chain x01–x08
Kept for provenance; not needed on the happy path. Each cell rejected one hypothesis:
x01 template mismatch → x02 adapter loading → x03 truncation → x04 completion learning
→ x05 boundary tokens (localized the pathology) → x06 TRL rendering → x07 trainer labels
→ x08 dynamics + checkpoint audit ⇒ root cause K (untrained think-opener under Base + LoRA).


In [ ]:
# @title x01_diagnose — chat-template consistency (hypothesis A)
!python scripts/build_training_data.py

!python scripts/inspect_sft_tokenization.py --config configs/experiments/eval_playworld.yaml


{
  "seed": 1042,
  "sft_records": 2000,
  "prompt_records": 2000,
  "unsolvable_dropped": 0,
  "sft_fingerprint": "sha256:9d4a5f494dd85c1edf0f6e4bfcac24e120bbf8b6b6386f5282b320002b683838",
  "prompt_fingerprint": "sha256:cc2aef0df4f5efda3db7ccfd43c76e40260935e9b7b55ee5760490e6a4304f14",
  "train_families": [
    "train-fam0",
    "train-fam1",
    "train-fam2",
    "train-fam3",
    "train-fam4"
  ],
  "eval_families_checked": [
    "eval_adversarial-fam0",
    "eval_comp_ood-fam0",
    "eval_comp_ood-fam1",
    "eval_id-fam0",
    "eval_id-fam1",
    "eval_id-fam2",
    "eval_rule_ood-fam0",
    "eval_rule_ood-fam1",
    "eval_template_ood-fam0",
    "eval_template_ood-fam1"
  ]
}
config.json: 100% 729/729 [00:00<00:00, 2.11MB/s]
tokenizer_config.json: 100% 9.68k/9.68k [00:00<00:00, 29.7MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 28.8MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 26.3MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 213MB/s]
chat_template defined: True
eos:

In [ ]:
# @title x02_adapter_effect — logit delta (hypothesis B)
RUN_ID = "20260801-030335--a1-playworld-sft--s42--e24d72"
out = !python scripts/fetch_run.py --repo m97j/aw-runs-a1 --run-id {RUN_ID}
print("\n".join(out))
adapter_dir = [l for l in out if l.startswith("ADAPTER_DIR=")][0].split("=", 1)[1]

!python scripts/diag_adapter_effect.py \
  --config configs/experiments/eval_playworld.yaml \
  --adapter-dir {adapter_dir}


local artifacts reused (sha256 verified: sha256:f84fb34da194...)
ADAPTER_DIR=runs/20260801-030335--a1-playworld-sft--s42--e24d72/artifacts/final_adapter
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Loading weights: 100% 399/399 [00:01<00:00, 339.46it/s]
first-step logit delta: mean=17.747099 max=34.562500
outputs identical: False
--- BASE OUTPUT (first 800 chars) ---
{"actions": [{"type": "COLLECT", "resource": "res_1"}], "final_state": {"location": "loc_3", "energy": 4, "inventory": {"res_1": 1}, "deposited": {}, "turn": 1}}<|endoftext|>
--- ADAPTER OUTPUT (first 800 chars) ---
.Aggressive exploration and resource collection s

In [ ]:
# @title x03_diag_token_lengths — truncation audit (hypothesis C)
!python scripts/diag_token_lengths.py \
    --config configs/experiments/eval_playworld.yaml \
    --prompt-file data/train/playworld_sft.jsonl \
    --caps 1024 4096


records tokenized: 2000  (file: data/train/playworld_sft.jsonl)
FULL length  min/p50/p90/p99/max: 333/391/430/476/516
PROMPT length min/p50/p90/p99/max: 288/305/310/313/314
--- cap=1024 ---
  truncated at all      : 0/2000 (0.0%)
  completion FULLY lost : 0/2000 (0.0%)
  completion partly lost: 0/2000 (0.0%)
--- cap=4096 ---
  truncated at all      : 0/2000 (0.0%)
  completion FULLY lost : 0/2000 (0.0%)
  completion partly lost: 0/2000 (0.0%)
VERDICT: hypothesis C NOT supported — truncation at the legacy cap is negligible; investigate loss masking / label construction next.


In [ ]:
# @title x04_diag_completion_learning — span-split accuracy (hypothesis D)
!python scripts/diag_completion_learning.py \
    --config configs/experiments/eval_playworld.yaml \
    --adapter-dir runs/20260729-145835--a1-playworld-sft--s42--1eb4c7/artifacts/final_adapter \
    --prompt-file data/train/playworld_sft.jsonl \
    --num-samples 8


config.json: 100% 729/729 [00:00<00:00, 6.41MB/s]
tokenizer_config.json: 100% 9.68k/9.68k [00:00<00:00, 32.2MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 74.5MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 76.6MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 87.1MB/s]
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
model.safetensors.index.json: 100% 32.9k/32.9k [00:00<00:00, 178MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0% 0/5 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/7.15G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   

In [ ]:
# @title x05_diag_first_token — boundary dissection (hypotheses E/F)
!python scripts/diag_first_token.py \
    --config configs/experiments/eval_playworld.yaml \
    --adapter-dir {adapter_dir} \
    --prompt-file data/train/playworld_sft.jsonl \
    --num-samples 4


Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Loading weights: 100% 399/399 [00:01<00:00, 338.45it/s]
[BASE] target first tokens: ['<think>', '\n\n', '</think>', '\n\n', '{"', 'actions', '":', ' [{"']
[BASE] top-5 @ TRAIN-slice boundary : [('{"', 0.2533904016017914), ('```', 0.17415250837802887), ('{\n', 0.13563011586666107), ('The', 0.04403264820575714), ('Here', 0.03885867819190025)]
[BASE] top-5 @ EVAL-encode boundary : [('{"', 0.2533904016017914), ('```', 0.17415250837802887), ('{\n', 0.13563011586666107), ('The', 0.04403264820575714), ('Here', 0.03885867819190025)]
[BASE] forced k=1: '<think>\n```json\n{\n  "actions": [\n  

In [ ]:
# @title x06_diag_trl_rendering — TRL vs eval rendering (hypothesis G, CPU-ok)
!python scripts/diag_trl_rendering.py \
    --config configs/experiments/eval_playworld.yaml \
    --prompt-file data/train/playworld_sft.jsonl \
    --num-samples 4


config.json: 100% 729/729 [00:00<00:00, 8.18MB/s]
tokenizer_config.json: 100% 9.68k/9.68k [00:00<00:00, 35.7MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 76.7MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 256MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 310MB/s]
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
record 0: ours == trl-rendered : True
  '<think>' in ours rendering : True
  '<think>' in trl rendering  : True
  '<think>' in eval prompt    : False
  eval prompt tail            : 'final_state": {"location": "...", "energy": N}}<|im_end|>\n<|im_start|>assistant\n'
  assistant opener region     : 'POSIT|WAIT", ..

In [ ]:
# @title x07_diag_trainer_labels — processed tensors & labels (hypothesis H, CPU-ok)
!python scripts/diag_trainer_labels.py \
    --config configs/experiments/a1_playworld_sft.yaml \
    --prompt-file data/train/playworld_sft.jsonl \
    --num-samples 4


Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
Tokenizing train dataset: 100% 4/4 [00:00<00:00, 250.28 examples/s]
Building labels for train dataset: 100% 4/4 [00:00<00:00, 1880.64 examples/s]
Truncating train dataset: 100% 4/4 [00:00<00:00, 1928.42 examples/s]
Dropping fully masked examples from train dataset: 100% 4/4 [00:00<00:00, 2748.56 examples/s]
processed dataset columns: ['messages', 'input_ids', 'labels']
'<think>' token id(s): [151667]   '</think>': [151668]
example length: 408   has labels: True
[PROCESSED] window around

In [ ]:
# @title x08_diag_training_dynamics — overfit probe + checkpoint audit (hypotheses I/J)
!python scripts/diag_training_dynamics.py \
    --config configs/experiments/a1_playworld_sft.yaml \
    --prompt-file data/train/playworld_sft.jsonl \
    --adapter-dir runs/20260729-145835--a1-playworld-sft--s42--1eb4c7/artifacts/final_adapter \
    --steps 60


config.json: 100% 729/729 [00:00<00:00, 8.40MB/s]
tokenizer_config.json: 100% 9.68k/9.68k [00:00<00:00, 33.5MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 28.0MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 175MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 249MB/s]
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
Tokenizing train dataset: 100% 4/4 [00:00<00:00, 249.09 examples/s]
Building labels for train dataset: 100% 4/4 [00:00<00:00, 1912.59 examples/s]
Truncating train dataset: 100% 4/4 [00:00<00:00, 1895.09 examples/s]
D